# LLM Dynamo on Azure AKS - Autoscaling with KEDA

## Overview
End-to-end runbook to deploy the NVIDIA Dynamo inference platform on an existing AKS cluster and enable KEDA autoscaling driven by **Time-to-First-Token (TTFT)** latency metrics from Prometheus.

> **Infrastructure prerequisite:** The AKS cluster, system pool, and GPU node pool must already exist inside the CloudLabs-owned resource group. Provision them once with the ARM template `aks-dynamo.json` + `aks-dynamo.parameters.json` co-located with this notebook:
>
> ```
> az deployment group create \
>   --resource-group <CloudLabs-RG> \
>   --template-file ./aks-dynamo.json \
>   --parameters ./aks-dynamo.parameters.json
> ```

## Inference Architecture

```
Client → LoadBalancer (port 8000)
           └─ Frontend pods ×2      (vllm-runtime 0.9.0, metrics :8000)
                   └─ VllmDecodeWorker pods ×2–4  (vllm-runtime 0.9.0, metrics :9090)
                              Model: Qwen/Qwen3-0.6B via dynamo.vllm
```

## Autoscaling Signal
KEDA polls Prometheus every **30 s**. When the **TTFT p95 > 300 ms** (2-minute rolling window), it scales up `VllmDecodeWorker` replicas from a minimum of **2** to a maximum of **4**. Scale-down cooldown is **120 s**.

## Prerequisites
- Azure CLI installed and authenticated against the subscription that owns the AKS cluster
- AKS cluster already deployed via `aks-dynamo.json` (system pool and GPU `gpupool` ready)
- GPU quota for the GPU VM size used by `gpupool` (e.g. `Standard_NV36ads_A10_v5`, `Standard_NC40ads_H100_v5`)

## Steps

| Step | Description |
|------|-------------|
| 1 | Azure Login & Subscription Selection |
| 2 | Environment Configuration |
| 3 | Import AKS Credentials & List Cluster / Nodes |
| 4 | Install Cluster Tooling (kubectl, Helm, GPU Operator) |
| 5 | Deploy Prometheus Monitoring Stack |
| 6 | Install NVIDIA Dynamo Platform |
| 7 | Install KEDA & Apply Azure Monitor Prometheus Scraping Config |
| 8 | Deploy LLM Inference Workload |
| 9 | Apply KEDA Autoscaling Policy (TTFT-based ScaledObject) |
| 10 | Expose the Frontend Service |
| 11 | Smoke Test & Load-Driven Autoscaling Validation |
| 12 | Observe Autoscaling Behavior & TTFT P95 Results |
| 13 | Azure Managed Grafana Dashboard for VLLM |


## 1) Azure Login & Subscription Selection

Upgrade the Azure CLI, sign in, and pin the subscription you want to deploy into. Set `SUBSCRIPTION_ID` below (or leave blank to keep your current default subscription).

In [ ]:
# Upgrade Azure CLI to the latest release.
# Ensures the aks, deployment, monitor, and acr sub-commands used throughout this notebook are available.
!az upgrade --yes

In [ ]:
# Sign in.
# - On an Azure compute instance with a managed identity:  az login --identity
# - On a local machine:                                    az login        (interactive browser)
# - On a headless / SSH host:                              az login --use-device-code
!az login

In [ ]:
import os

# Set the subscription used for all subsequent commands.
# Replace with your subscription ID (GUID) or leave the empty string to keep your CLI default.
os.environ["SUBSCRIPTION_ID"] = ""

if os.environ["SUBSCRIPTION_ID"]:
    !az account set --subscription {os.environ['SUBSCRIPTION_ID']}

# Print the active subscription so you can confirm before any resource is created.
!az account show --output table

## 2) Environment Configuration

Set `RESOURCE_GROUP` to the CloudLabs-owned resource group that holds the AKS cluster. `CLUSTER_NAME` is fixed to match the ARM template.

| Variable | Value | Description |
|----------|-------|-------------|
| `RESOURCE_GROUP` | (your CloudLabs RG) | The resource group CloudLabs deployed the ARM template into |
| `CLUSTER_NAME` | `dynamo-cluster` | Fixed — matches the ARM `clusterName` variable |
| `REGION` | `indonesiacentral` | Azure region (informational) |


In [ ]:
import os

# Set to the CloudLabs-owned resource group that contains the AKS cluster.
os.environ["RESOURCE_GROUP"] = "aks_dynamo"
os.environ["CLUSTER_NAME"]   = "dynamo-cluster"
os.environ["REGION"]         = "indonesiacentral"

print("RESOURCE_GROUP:", os.environ["RESOURCE_GROUP"])
print("CLUSTER_NAME  :", os.environ["CLUSTER_NAME"])
print("REGION        :", os.environ["REGION"])


## 3) Import AKS Credentials & List Cluster / Nodes

1. `az aks list` confirms the cluster shows up under the active subscription.
2. `az aks nodepool list` shows pool details (count, VM size, autoscaler bounds, zones).
3. `az aks get-credentials` merges the AKS kubeconfig into `~/.kube/config`.
4. `kubectl get nodes` confirms both the system pool and `gpupool` nodes are `Ready`. Look for `agentpool=gpupool` on the GPU nodes.


In [ ]:
# List AKS clusters in the resource group and across the subscription.
!az aks list --resource-group {os.environ['RESOURCE_GROUP']} --output table
!az aks list --output table

In [ ]:
# Show node pool details from the Azure control plane (count, VM size, autoscaler bounds, zones).
!az aks nodepool list \
  --resource-group {os.environ['RESOURCE_GROUP']} \
  --cluster-name {os.environ['CLUSTER_NAME']} \
  --output table

In [ ]:
# Import AKS kubeconfig.
# --overwrite-existing: safe to re-run if credentials have rotated.
!az aks get-credentials --overwrite-existing --resource-group {os.environ['RESOURCE_GROUP']} --name {os.environ['CLUSTER_NAME']}

In [ ]:
# Confirm kubectl can talk to the cluster and that all nodes are Ready.
# Expect <CPU_COUNT> system-pool nodes + <AKS_MIN_NODE_COUNT> gpupool nodes.
!kubectl get nodes -o wide --show-labels

## 4) Install Cluster Tooling

- **`kubectl`** — installed here if running in Azure Cloud Shell or a bare Linux VM that may lack it.
- **Helm 3** — required for all platform charts (GPU Operator, Prometheus, Dynamo, KEDA).
- **NVIDIA GPU Operator** — installed immediately after Helm, before any workloads, so that GPU nodes initialise drivers and device plugins before Dynamo pods schedule onto them.


In [ ]:
# kubectl installation notes:
# - On Windows: kubectl is installed automatically with Azure CLI (az aks install-cli)
# - On macOS: brew install kubectl
# - On Linux: sudo apt-get install -y kubectl
# 
# For Windows, run this instead:
!az aks install-cli

In [ ]:
import sys
import subprocess

# --- Prerequisites ---
try:
    import requests
except ImportError:
    print("Installing required package: requests")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "requests"])
    import requests

import shutil
import os
import zipfile

HELM_DIR = os.path.join(os.path.expanduser("~"), ".local", "helm")

def find_helm():
    path = shutil.which("helm")
    if path:
        return path
    candidate = os.path.join(HELM_DIR, "windows-amd64", "helm.exe")
    return candidate if os.path.isfile(candidate) else None

def install_helm():
    print("Helm not found. Fetching latest release info...")
    resp = requests.get("https://api.github.com/repos/helm/helm/releases/latest")
    resp.raise_for_status()
    tag = resp.json()["tag_name"]  # e.g. v3.15.3

    zip_name = f"helm-{tag}-windows-amd64.zip"
    download_url = f"https://get.helm.sh/{zip_name}"
    os.makedirs(HELM_DIR, exist_ok=True)
    zip_path = os.path.join(HELM_DIR, zip_name)

    print(f"Downloading {download_url} ...")
    with requests.get(download_url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(HELM_DIR)
    os.remove(zip_path)

    helm_exe = os.path.join(HELM_DIR, "windows-amd64", "helm.exe")
    if not os.path.isfile(helm_exe):
        raise RuntimeError("helm.exe not found after extraction")

    os.environ["PATH"] = os.path.dirname(helm_exe) + os.pathsep + os.environ["PATH"]
    print(f"Installed to {helm_exe}")
    return helm_exe

helm_path = find_helm()
if helm_path is None:
    try:
        helm_path = install_helm()
    except Exception as e:
        print(f"Automatic install failed: {e}")
        print("Install manually instead:")
        print("  choco install kubernetes-helm")
        print("  scoop install helm")
        print("  or download from https://github.com/helm/helm/releases")
        helm_path = None

if helm_path:
    result = subprocess.run([helm_path, "version", "--short"], capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())

In [ ]:
# Install the NVIDIA GPU Operator into the gpu-operator namespace.
# The operator installs the CUDA driver, container toolkit, device plugin, and DCGM exporter
# on nodes that carry the gpu-driver=none label set during nodepool creation.
!helm repo add nvidia https://helm.ngc.nvidia.com/nvidia --force-update
!helm repo update
!helm upgrade --install gpu-operator nvidia/gpu-operator --namespace gpu-operator --create-namespace --version v26.3.2 --wait

## 5) Deploy Prometheus Monitoring Stack

Install `kube-prometheus-stack` in the `monitoring` namespace. Two Helm values override the default namespace-scoped `PodMonitor` selectors so Prometheus can scrape pods in **any** namespace:

```
podMonitorSelectorNilUsesHelmValues=false   → disable namespace-scoped selector filter
podMonitorNamespaceSelector.matchLabels=null → watch all namespaces for PodMonitor objects
probeNamespaceSelector.matchLabels=null      → same for Probe resources
```

Without these settings, Prometheus would only scrape pods with monitors inside the `monitoring` namespace and would miss the `dynamo-cloud` workload metrics that KEDA reads.

The KEDA trigger in `autoscale_ttf.yaml` points directly at this Prometheus service:
`http://prometheus-kube-prometheus-prometheus.monitoring.svc.cluster.local:9090`


In [ ]:
# Install kube-prometheus-stack in the dedicated 'monitoring' namespace.
# The two selector overrides allow PodMonitors and Probes from OTHER namespaces (e.g. dynamo-cloud)
# to be discovered by this Prometheus instance, which is required for KEDA's Prometheus trigger.
!helm uninstall prometheus -n monitoring
!helm upgrade --install prometheus -n monitoring --create-namespace prometheus-community/kube-prometheus-stack \
  --set prometheus.prometheusSpec.podMonitorSelectorNilUsesHelmValues=false \
  --set-json prometheus.prometheusSpec.podMonitorNamespaceSelector={} \
  --set-json prometheus.prometheusSpec.probeNamespaceSelector={}

## 6) Install NVIDIA Dynamo Platform

After installation, verify all pods in `dynamo-system` are `Running` before deploying the inference workload.


In [ ]:
NAMESPACE = "dynamo-system"  # don't change
RELEASE_VERSION = "1.1.0-dev.1"

chart_url = f"https://helm.ngc.nvidia.com/nvidia/ai-dynamo/charts/dynamo-platform-{RELEASE_VERSION}.tgz"

!helm fetch {chart_url}
!helm upgrade --install dynamo-platform dynamo-platform-{RELEASE_VERSION}.tgz --namespace {NAMESPACE} --create-namespace

In [ ]:
NAMESPACE = "dynamo-system"

!kubectl get pods -n {NAMESPACE}

**<mark>Make sure all are Ready before going to next </mark>**

## 7) Install KEDA 

**KEDA** (installed in `keda` namespace):
- `--set metricsServer.enabled=false` — disables the built-in metrics server since we route through Prometheus instead.
- KEDA operator watches `ScaledObject` resources and adjusts HPA `desiredReplicas` based on external metrics.


In [ ]:
# Install KEDA in its own namespace.
# metricsServer.enabled=false: we rely on Prometheus for metrics rather than the KEDA metrics server.
!helm repo add kedacore https://kedacore.github.io/charts --force-update
!helm repo update
!helm upgrade --install keda kedacore/keda --namespace keda --create-namespace

In [ ]:
# Confirm the keda-operator and keda-operator-metrics-apiserver pods are Running.
!kubectl get pods -n keda

**<mark>Make sure all are Ready before going to next </mark>**

### Apply Azure Monitor Prometheus Scraping Config
**`config_prome.yaml`** (`ConfigMap` → `kube-system/ama-metrics-settings-configmap`):
- Applies to the Azure Monitor Metrics agent (AMA), configuring which scrapers are enabled.
- Crucially enables **pod-annotation-based scraping** scoped to the `dynamo-cloud` namespace:
  ```
  podannotationnamespaceregex = "dynamo-cloud"
  ```
- This causes the AMA agent to discover and scrape pods that carry `prometheus.io/scrape: "true"` annotations — which both `Frontend` (port 8000) and `VllmDecodeWorker` (port 9090) pods have in `agg.yaml`.

In [ ]:
# Apply the Azure Monitor Managed Prometheus configuration (ConfigMap in kube-system).
# Key settings in config_prome.yaml:
#   - pod-annotation-based scraping scoped to the 'dynamo-cloud' namespace
#   - cadvisor, kubestate, nodeexporter enabled at 30 s intervals
#   - dcgmexporter disabled (GPU metrics come via DCGM inside the GPU Operator instead)
!kubectl apply -f ./config_prome.yaml

## 8) Deploy LLM Inference Workload

Apply `agg.yaml` which creates a `DynamoGraphDeployment` with:

| Component | Replicas | Image | Metrics Port |
|-----------|----------|-------|-------------|
| `Frontend` | 2 | `dynamovllm.azurecr.io/vllm-runtime:0.9.0` | 8000 |
| `VllmDecodeWorker` | 2 (scales 2→4) | `dynamovllm.azurecr.io/vllm-runtime:0.9.0` | 9090 |

The model served is **Qwen/Qwen3-0.6B** via the `dynamo.vllm` Python module. Both components carry `prometheus.io/scrape: "true"` pod annotations that are picked up by the `config_prome.yaml` scraping configuration applied in step 7.


In [ ]:
# Create the inference workload namespace.
!kubectl create namespace dynamo-cloud

# Apply agg.yaml which creates:
#   - DynamoGraphDeployment vllm-agg:
#       Frontend (2 replicas, port 8000, scrape: true)
#       VllmDecodeWorker (2 replicas, port 9090, scalingAdapter enabled)
# The Dynamo controller reconciles this into Deployments + Services automatically.
!kubectl apply -f ./agg.yaml -n dynamo-cloud

In [ ]:
import os

# Wait for Frontend and VllmDecodeWorker pods to reach Running state.
# Expected: 2x vllm-agg-frontend, 2x vllm-agg-vllmdecodeworker
namespace = "dynamo-cloud"
!kubectl get pods -n {namespace}

**<mark>Make sure all are Ready before going to next </mark>**

In [ ]:
# List DynamoGraphDeploymentScalingAdapters (dgdsa).
# KEDA's ScaledObject targets 'vllm-agg-vllmdecodeworker' of this kind.
# This adapter must exist and be READY before applying the ScaledObject.
!kubectl get dgdsa -n dynamo-cloud

## 9) Apply KEDA Autoscaling Policy (TTFT-based ScaledObject)

`autoscale_ttf.yaml` creates the KEDA `ScaledObject` `vllm-agg-decode-scaler`, which targets the `DynamoGraphDeploymentScalingAdapter` created automatically for `VllmDecodeWorker`:

| Parameter | Value |
|-----------|-------|
| `scaleTargetRef` | `vllm-agg-vllmdecodeworker` (kind: `DynamoGraphDeploymentScalingAdapter`) |
| Replica range | min **2** → max **4** |
| Poll interval | 30 s |
| Cooldown | 120 s before scale-down |
| Trigger | Prometheus — `dynamo_frontend_time_to_first_token_seconds_bucket` |
| PromQL | `histogram_quantile(0.95, sum(rate(...[2m])) by (le))` |
| Scale-up threshold | **0.3 s (300 ms)** TTFT p95 |

> **Pre-requisite:** Confirm `kubectl get dgdsa -n dynamo-cloud` shows the adapter as `READY` before applying this manifest.


In [ ]:
# Apply autoscale_ttf.yaml which creates the KEDA ScaledObject 'vllm-agg-decode-scaler'.
# Scaling logic:
#   trigger: Prometheus query -> histogram_quantile(0.95, rate(dynamo_frontend_time_to_first_token_seconds_bucket[2m]))
#   threshold: 0.3 s (300 ms) -> scale up VllmDecodeWorker
#   pollingInterval: 30 s  |  cooldownPeriod: 120 s  |  range: 2-4 replicas
!kubectl apply -f ./autoscale_ttf.yaml -n dynamo-cloud

In [ ]:
# Confirm the ScaledObject is Active and targeting the correct DynamoGraphDeploymentScalingAdapter.
# STATUS=Active means KEDA has connected to Prometheus and the scaler is healthy.
!kubectl get scaledobject -n dynamo-cloud

## 10) Expose the Frontend Service

Patch `vllm-agg-frontend` from `ClusterIP` to `LoadBalancer`. Azure provisions a public IP and routes traffic on port **8000** to the Frontend pods, which expose the OpenAI-compatible `/v1/chat/completions` API.

> Wait until `kubectl get svc` shows a real IP in `EXTERNAL-IP` (not `<pending>`) before running the smoke test. This typically takes 1–2 minutes.


In [ ]:
# Patch the Frontend service from ClusterIP to LoadBalancer.
# This provisions an Azure public load balancer on port 8000 routed to the Frontend pods.
# The endpoint exposes the OpenAI-compatible /v1/chat/completions API.
import json

namespace = "dynamo-cloud"

patch = {
    "spec": {
        "type": "LoadBalancer",
        "ports": [
            {"port": 8000, "targetPort": 8000}
        ]
    }
}

with open("frontend-patch.json", "w") as f:
    json.dump(patch, f)

!kubectl patch svc vllm-agg-frontend -n {namespace} --patch-file frontend-patch.json

## 11) Smoke Test 

**Smoke test** — single `curl` to the OpenAI-compatible `/v1/chat/completions` endpoint, confirming the service is reachable and Qwen/Qwen3-0.6B is loaded.


In [ ]:
import os

# Get the external IP assigned by Azure to the LoadBalancer.
# Wait until EXTERNAL-IP is no longer '<pending>' before running the smoke test.
# Provisioning typically takes 1-2 minutes.
namespace = "dynamo-cloud"
!kubectl get svc vllm-agg-frontend -n {namespace}

In [ ]:
import subprocess
import sys
import time

try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "requests"])
    import requests

NAMESPACE = "dynamo-cloud"
SERVICE = "vllm-agg-frontend"

def get_external_ip(namespace, service, retries=10, delay=5):
    for _ in range(retries):
        result = subprocess.run(
            ["kubectl", "get", "svc", service, "-n", namespace,
             "-o", "jsonpath={.status.loadBalancer.ingress[0].ip}"],
            capture_output=True, text=True,
        )
        ip = result.stdout.strip()
        if ip:
            return ip
        time.sleep(delay)
    raise RuntimeError(
        f"No external IP found for service '{service}' in namespace '{namespace}' "
        "after waiting — the LoadBalancer may still be provisioning."
    )

external_ip = get_external_ip(NAMESPACE, SERVICE)
print(f"Using external IP: {external_ip}")

payload = {
    "model": "Qwen/Qwen3-0.6B",
    "messages": [
        {"role": "user", "content": [{"type": "text", "text": "hi?"}]}
    ],
    "stream": False,
}

response = requests.post(
    f"http://{external_ip}:8000/v1/chat/completions",
    json=payload,
    timeout=30,
)
response.raise_for_status()
data = response.json()

assert "choices" in data, f"No 'choices' in response: {data}"
print("Model reply:", data["choices"][0]["message"]["content"])

### Load-Driven Autoscaling Validation

In [ ]:
# Install aiperf — NVIDIA's LLM inference performance profiler.
# Provides request-level latency (TTFT, TPOT, E2E), throughput, and token stats.
!pip install -v aiperf


**Load test** — `aiperf profile` drives sustained high-rate traffic to push TTFT p95 above the 300 ms KEDA threshold:

| aiperf parameter | Value | Notes |
|-----------------|-------|-------|
| `--model` | `Qwen/Qwen3-0.6B` | Must match the `--model` arg in `agg.yaml` |
| `--endpoint-type` | `chat` | OpenAI chat completions format |
| `--synthetic-input-tokens-mean` | 3000 | Long-context requests to stress the decode workers |
| `--output-tokens-mean` | 250 | Typical completion length |
| `--request-rate` | 42 req/s | High enough to push TTFT p95 past 300 ms |
| `--request-count` | 64800 | KEDA polling period is 30 s |
| `--artifact-dir` | `/tmp/scaling_test_phase2_extended` | Saved JSON latency and throughput results |

**While the test runs**, monitor autoscaling from a separate terminal:
```bash
kubectl get pods -n dynamo-cloud -w          # watch new VllmDecodeWorker pods appear
```

**<mark>Run this in the terminal</mark>**

In [ ]:
# Load test: generate 64800 requests at 42 req/s with 3000-token inputs.
# At this rate TTFT p95 is expected to exceed 300 ms, which triggers KEDA to add decode workers.
# While this runs, run in a separate terminal to observe autoscaling:
#   kubectl get pods -n dynamo-cloud -w
#   kubectl get hpa -n dynamo-cloud -w
import os
import subprocess
import sys
import time

try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "requests"])
    import requests

NAMESPACE = "dynamo-cloud"
SERVICE = "vllm-agg-frontend"

def get_external_ip(namespace, service, retries=10, delay=5):
    for _ in range(retries):
        result = subprocess.run(
            ["kubectl", "get", "svc", service, "-n", namespace,
             "-o", "jsonpath={.status.loadBalancer.ingress[0].ip}"],
            capture_output=True, text=True,
        )
        ip = result.stdout.strip()
        if ip:
            return ip
        time.sleep(delay)
    raise RuntimeError(
        f"No external IP found for service '{service}' in namespace '{namespace}' "
        "after waiting — the LoadBalancer may still be provisioning."
    )

external_ip = get_external_ip(NAMESPACE, SERVICE)
print(f"Using external IP: {external_ip}")
os.environ["EXTERNAL_IP"] = external_ip

os.makedirs(os.path.expanduser("~/scaling_test_phase2_extended"), exist_ok=True)

!aiperf profile \
  --model Qwen/Qwen3-0.6B \
  --tokenizer Qwen/Qwen3-0.6B \
  --endpoint-type chat \
  --url http://$EXTERNAL_IP:8000 \
  --streaming \
  --synthetic-input-tokens-mean 3000 \
  --output-tokens-mean 250 \
  --request-rate 42 \
  --request-count 64800 \
  --num-dataset-entries 18000 \
  --artifact-dir ~/scaling_test_phase2_extended

## 12) Observed Autoscaling Behavior — TTFT P95 Results

The chart below captures **TTFT P95 (Time-to-First-Token, 95th percentile)** in seconds over ~50 minutes while KEDA scaled `VllmDecodeWorker` between **2 and 4 NC-H100 nodes**.

> **Important — load profile:** The same constant request rate (36 req/s) is maintained **throughout the entire test**. There is no change in incoming traffic. All TTFT fluctuations are driven purely by the number of active decode workers, not by changes in load.

![TTF P95 – KEDA Scaler Agg from 2 to 4 NC-H100](img/ttf.png)

---

### Event Timeline

| Time | Event | TTFT P95 Behavior |
|------|-------|-------------------|
| **~14:35** | **First scale-up triggered** | Constant load saturates the 2 baseline workers. KEDA detects TTFT p95 > 300 ms and requests 2 additional `VllmDecodeWorker` pods. |
| **14:35 → 14:40** | **Cold node provisioning (~10 min)** | New GPU nodes must be provisioned by Azure, then the GPU Operator installs drivers and Dynamo loads the model. TTFT continues rising during this entire window — no new decode capacity is available yet. |
| **~14:40** | **New pods start running** | The 2 additional workers come online. 4 GPUs now handle the same constant load — TTFT p95 drops sharply. |
| **~14:52** | **First scale-down attempt** | With 4 workers the TTFT p95 falls below 300 ms, so KEDA scales back to 2 workers after the 120 s cooldown. **Traffic is unchanged.** The 2 remaining workers are immediately overloaded and TTFT spikes again. |
| **~14:56** | **Second scale-up** | KEDA re-triggers a scale-up. This completes **fast** because AKS has not yet de-provisioned the extra nodes — the nodes are still running and warm (driver loaded, model cached in GPU VRAM). TTFT drops quickly. |
| **~15:02** | **Second scale-down attempt** | Same cycle — KEDA cuts back to 2 workers once TTFT drops below threshold. Constant traffic again overwhelms the reduced capacity, TTFT rises. |
| **~15:06** | **Third scale-up** | Fast again for the same reason: AKS nodes are still provisioned and warm. TTFT spike is shorter than the first cold-start event. |

> **Note on warm vs. cold scale-ups:** The second and third scale-ups are fast **only because AKS has not yet terminated the extra nodes**. AKS node scale-down has its own separate delay (typically several minutes after pods are removed). If the nodes had been terminated between events, the next scale-up would incur the same ~10-minute cold-start penalty as the first.


## 13) Azure Managed Grafana Dashboard 


From the AKS cluster view on Azure portal (`Dashboards with Grafana section`) add and import VLLM Grafana dashboard for monitoring. This dashboard is designed to visualize key performance metrics of the VLLM model deployments, such as latency, throughput, and resource utilization.


Json Grafana dashboard file to import:
https://grafana.com/grafana/dashboards/24756-vllm-monitoring-v2/


![vllm dashboard](img/vllm_dashboard.png)